# Contact Data Web Scraper and Enricher

This notebook enriches `data/contacts.csv` using live web search evidence (SerpAPI, with Tavily fallback) and optional LLM-based structured extraction.

It is designed to be run top-to-bottom. The output is a final enriched table containing all original columns plus metadata, source links, confidence, and timestamps.

## 1) Install and Import Libraries

Run this cell first if your kernel does not already have dependencies.

In [1]:
# Uncomment if needed
# %pip install -q pandas requests python-dotenv tenacity tqdm openai

import os
import re
import json
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd
import requests
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm.auto import tqdm

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

from IPython.display import display

c:\Users\KISHORE S\anaconda3\envs\agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Configure Secrets and Runtime Parameters

Set API keys in environment or `.env`.

Priority:
1. `SERPAPI_API_KEY` for Google search via SerpAPI
2. `TAVILY_API_KEY` fallback if SerpAPI key is missing
3. Optional LLM extraction via OpenRouter using `OPENROUTER_API_KEY`

In [2]:
load_dotenv()

CONFIG: Dict[str, Any] = {
    "input_csv": "data/contacts.csv",
    "output_csv": "data/contacts_enriched.csv",
    "output_parquet": "data/contacts_enriched.parquet",
    "evidence_json": "data/contacts_enrichment_evidence.json",
    "unresolved_csv": "data/contacts_unresolved.csv",
    "checkpoint_csv": "data/contacts_enriched_checkpoint.csv",
    "cache_dir": "data/.cache_contacts_scraper",
    "max_results": 6,
    "timeout_sec": 25,
    "max_retries": 3,
    "batch_size": 5,
    "min_confidence_to_update": 0.60,
}

API_KEYS = {
    "serpapi": os.getenv("SERPAPI_API_KEY", "").strip(),
    "tavily": os.getenv("TAVILY_API_KEY", "").strip(),
    "openrouter": os.getenv("OPENROUTER_API_KEY", "").strip(),
}

Path(CONFIG["cache_dir"]).mkdir(parents=True, exist_ok=True)

print("SerpAPI key available:", bool(API_KEYS["serpapi"]))
print("Tavily key available:", bool(API_KEYS["tavily"]))
print("OpenRouter key available:", bool(API_KEYS["openrouter"]))

SerpAPI key available: False
Tavily key available: True
OpenRouter key available: True


## 3) Load contacts.csv and Validate Column Schema

In [3]:
required_columns = [
    "name", "company", "role", "type", "industry", "email", "linkedin", "relevance_score"
]

input_path = Path(CONFIG["input_csv"])
if not input_path.exists():
    raise FileNotFoundError(f"CSV not found: {input_path}")

contacts_df = pd.read_csv(input_path)
contacts_df.columns = [c.strip() for c in contacts_df.columns]

missing_cols = [c for c in required_columns if c not in contacts_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

for col in contacts_df.columns:
    if contacts_df[col].dtype == "object":
        contacts_df[col] = contacts_df[col].fillna("").astype(str).str.strip()

contacts_df["relevance_score"] = pd.to_numeric(contacts_df["relevance_score"], errors="coerce").fillna(0.0)

print(f"Loaded {len(contacts_df)} rows and {len(contacts_df.columns)} columns")
display(contacts_df.head(5))

Loaded 20 rows and 8 columns


,name,company,role,type,industry,email,linkedin,relevance_score
0,Andrew Ng,Landing AI,CEO,speaker,AI,user0@example.com,https://linkedin.com/in/user0,0.77
1,Fei-Fei Li,Stanford,Professor,sponsor,Startup,user1@example.com,https://linkedin.com/in/user1,0.91
2,Demis Hassabis,DeepMind,CEO,sponsor,AI,user2@example.com,https://linkedin.com/in/user2,0.89
3,Anima Anandkumar,Caltech,Professor,sponsor,ML,user3@example.com,https://linkedin.com/in/user3,0.85
4,Satya Nadella,Microsoft,CEO,speaker,Tech,user4@example.com,https://linkedin.com/in/user4,0.73


## 4) Generate Search Queries Per Contact Row

In [4]:
def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def build_query(row: pd.Series) -> str:
    # Query is generated from row values dynamically; no fixed output assumptions.
    parts = []
    for key in ["name", "company", "role", "industry", "linkedin"]:
        if key in row.index and normalize_text(row[key]):
            parts.append(normalize_text(row[key]))
    parts.extend([
        "official profile",
        "company role",
        "linkedin",
        "email contact"
    ])
    return " ".join(parts)


def row_key(row: pd.Series) -> str:
    base = "|".join(normalize_text(row.get(c, "")) for c in contacts_df.columns)
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

preview_queries = contacts_df.head(3).apply(build_query, axis=1)
display(pd.DataFrame({"name": contacts_df.head(3)["name"], "query": preview_queries}))

,name,query
0,Andrew Ng,Andrew Ng Landing AI CEO AI https://linkedin.c...
1,Fei-Fei Li,Fei-Fei Li Stanford Professor Startup https://...
2,Demis Hassabis,Demis Hassabis DeepMind CEO AI https://linkedi...


## 5) Fetch Live Results with SerpAPI (Tavily Fallback)

Includes cache, retries, and timeout handling.

In [ ]:
cache_dir = Path(CONFIG["cache_dir"])


def _cache_path(namespace: str, key: str) -> Path:
    hashed = hashlib.md5(key.encode("utf-8")).hexdigest()
    return cache_dir / f"{namespace}_{hashed}.json"


def cache_get(namespace: str, key: str) -> Any:
    p = _cache_path(namespace, key)
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding="utf-8"))


def cache_set(namespace: str, key: str, value: Any) -> None:
    p = _cache_path(namespace, key)
    p.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


class APIError(Exception):
    pass


@retry(
    reraise=True,
    stop=stop_after_attempt(CONFIG["max_retries"]),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type((requests.RequestException, APIError)),
)
def fetch_serpapi_results(query: str, start: int = 0) -> List[Dict[str, str]]:
    if not API_KEYS["serpapi"]:
        raise APIError("SERPAPI_API_KEY not configured")

    params = {
        "engine": "google",
        "q": query,
        "api_key": API_KEYS["serpapi"],
        "num": min(10, int(CONFIG["max_results"])),
        "start": start,
    }
    resp = requests.get("https://serpapi.com/search.json", params=params, timeout=CONFIG["timeout_sec"])
    if resp.status_code >= 400:
        raise APIError(f"SerpAPI HTTP {resp.status_code}: {resp.text[:240]}")

    payload = resp.json()
    hits = []
    for item in payload.get("organic_results", []):
        hits.append({
            "title": normalize_text(item.get("title")),
            "url": normalize_text(item.get("link")),
            "snippet": normalize_text(item.get("snippet")),
            "source": "serpapi",
        })
    return hits


@retry(
    reraise=True,
    stop=stop_after_attempt(CONFIG["max_retries"]),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type((requests.RequestException, APIError)),
)
def fetch_tavily_results(query: str) -> List[Dict[str, str]]:
    if not API_KEYS["tavily"]:
        raise APIError("TAVILY_API_KEY not configured")

    body = {
        "api_key": API_KEYS["tavily"],
        "query": query,
        "max_results": int(CONFIG["max_results"]),
        "search_depth": "advanced",
        "include_answer": False,
        "include_raw_content": False,
    }
    resp = requests.post("https://api.tavily.com/search", json=body, timeout=CONFIG["timeout_sec"])
    if resp.status_code >= 400:
        raise APIError(f"Tavily HTTP {resp.status_code}: {resp.text[:240]}")

    payload = resp.json()
    hits = []
    for item in payload.get("results", []):
        hits.append({
            "title": normalize_text(item.get("title")),
            "url": normalize_text(item.get("url")),
            "snippet": normalize_text(item.get("content")),
            "source": "tavily",
        })
    return hits


def search_with_fallback(query: str) -> List[Dict[str, str]]:
    cached = cache_get("search", query)
    if cached is not None:
        return cached

    hits: List[Dict[str, str]] = []
    errors: List[str] = []

    if API_KEYS["serpapi"]:
        try:
            hits = fetch_serpapi_results(query)
        except Exception as exc:
            errors.append(f"serpapi: {exc}")

    if not hits and API_KEYS["tavily"]:
        try:
            hits = fetch_tavily_results(query)
        except Exception as exc:
            errors.append(f"tavily: {exc}")

    if not hits:
        hits = [{
            "title": "search_error",
            "url": "",
            "snippet": " | ".join(errors) if errors else "No search provider available",
            "source": "none",
        }]

    cache_set("search", query, hits)
    return hits


def build_evidence_text(hits: List[Dict[str, str]], max_chars: int = 5000) -> str:
    joined = []
    for idx, h in enumerate(hits, start=1):
        joined.append(
            f"[{idx}] title={h.get('title', '')}\nurl={h.get('url', '')}\nsnippet={h.get('snippet', '')}"
        )
    text = "\n\n".join(joined)
    return text[:max_chars]


def extract_email(text: str) -> str:
    m = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text or "")
    return m.group(0) if m else ""


def extract_linkedin(text: str) -> str:
    m = re.search(r"https?://(?:[a-z]{2,3}\.)?linkedin\.com/[^\s\)\]\,\"]+", text or "", flags=re.IGNORECASE)
    return m.group(0) if m else ""


def fallback_extract(row: pd.Series, evidence_text: str, hits: List[Dict[str, str]], schema_columns: List[str]) -> Tuple[Dict[str, Any], float]:
    out = {c: normalize_text(row.get(c, "")) for c in schema_columns}

    candidate_email = extract_email(evidence_text)
    candidate_linkedin = extract_linkedin(evidence_text)

    if "email" in out and candidate_email:
        out["email"] = candidate_email
    if "linkedin" in out and candidate_linkedin:
        out["linkedin"] = candidate_linkedin

    lower_evidence = evidence_text.lower()
    score = 0.45

    for key in ["name", "company", "role", "industry", "type"]:
        base = normalize_text(row.get(key, "")).lower()
        if base and base in lower_evidence:
            score += 0.08

    if candidate_email:
        score += 0.10
    if candidate_linkedin:
        score += 0.12

    score = max(0.0, min(0.95, score))
    return out, score


def llm_extract(row: pd.Series, evidence_text: str, schema_columns: List[str]) -> Tuple[Dict[str, Any], float, str]:
    if not API_KEYS["openrouter"] or OpenAI is None:
        fallback, score = fallback_extract(row, evidence_text, [], schema_columns)
        return fallback, score, "fallback"

    cache_key = hashlib.sha256((json.dumps(row.to_dict(), ensure_ascii=False) + evidence_text).encode("utf-8")).hexdigest()
    cached = cache_get("llm", cache_key)
    if cached is not None:
        return cached["fields"], float(cached["confidence"]), str(cached.get("mode", "llm_cached"))

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=API_KEYS["openrouter"],
    )

    schema_str = ", ".join(schema_columns)
    prompt = f"""
You are a data extraction system.
Extract and verify contact fields from web evidence.
Return strict JSON only with this shape:
{{
  "fields": {{... all columns as strings ...}},
  "confidence": 0.0,
  "notes": "short"
}}

Rules:
- Include every column exactly once: {schema_str}
- Preserve original values unless evidence strongly indicates correction.
- If unknown, keep original row value.
- confidence must be between 0 and 1.

Original row JSON:
{json.dumps(row.to_dict(), ensure_ascii=False)}

Evidence:
{evidence_text}
""".strip()

    try:
        response = client.chat.completions.create(
            model=os.getenv("LLM_MODEL", "openai/gpt-oss-120b"),
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )
        content = response.choices[0].message.content
        parsed = json.loads(content)
        fields = parsed.get("fields", {})
        confidence = float(parsed.get("confidence", 0.0))

        cleaned = {c: normalize_text(fields.get(c, row.get(c, ""))) for c in schema_columns}
        confidence = max(0.0, min(1.0, confidence))
        mode = "llm"
    except Exception:
        cleaned, confidence = fallback_extract(row, evidence_text, [], schema_columns)
        mode = "fallback_after_llm_error"

    cache_set("llm", cache_key, {"fields": cleaned, "confidence": confidence, "mode": mode})
    return cleaned, confidence, mode


def merge_with_confidence(original: pd.Series, extracted: Dict[str, Any], confidence: float, min_conf: float, schema_columns: List[str]) -> Dict[str, Any]:
    merged = {}
    for c in schema_columns:
        old_v = normalize_text(original.get(c, ""))
        new_v = normalize_text(extracted.get(c, old_v))

        if confidence >= min_conf and new_v:
            merged[c] = new_v
        else:
            merged[c] = old_v
    return merged


schema_columns = list(contacts_df.columns)
run_started_at = datetime.now(timezone.utc).isoformat()
all_rows: List[Dict[str, Any]] = []
evidence_log: List[Dict[str, Any]] = []
unresolved_rows: List[Dict[str, Any]] = []

for batch_start in range(0, len(contacts_df), int(CONFIG["batch_size"])):
    batch = contacts_df.iloc[batch_start: batch_start + int(CONFIG["batch_size"])]
    for idx, row in tqdm(batch.iterrows(), total=len(batch), desc=f"Batch {batch_start // CONFIG['batch_size'] + 1}"):
        query = build_query(row)
        hits = search_with_fallback(query)
        evidence_text = build_evidence_text(hits)
        extracted, confidence, mode = llm_extract(row, evidence_text, schema_columns)
        merged = merge_with_confidence(
            original=row,
            extracted=extracted,
            confidence=confidence,
            min_conf=float(CONFIG["min_confidence_to_update"]),
            schema_columns=schema_columns,
        )

        row_out = dict(merged)
        row_out["enrichment_confidence"] = round(float(confidence), 4)
        row_out["enrichment_mode"] = mode
        row_out["search_query"] = query
        row_out["scraped_at_utc"] = datetime.now(timezone.utc).isoformat()
        row_out["source_urls"] = " | ".join([h.get("url", "") for h in hits if h.get("url")])

        all_rows.append(row_out)

        evidence_log.append({
            "row_index": int(idx),
            "name": normalize_text(row.get("name", "")),
            "query": query,
            "confidence": confidence,
            "mode": mode,
            "hits": hits,
        })

        if confidence < float(CONFIG["min_confidence_to_update"]):
            unresolved_rows.append(row_out)

        time.sleep(0.25)

    pd.DataFrame(all_rows).to_csv(CONFIG["checkpoint_csv"], index=False)

enriched_df = pd.DataFrame(all_rows)
print("Enrichment complete.")
print("Rows:", len(enriched_df))
print("Unresolved rows:", len(unresolved_rows))

Batch 2:  60%|██████    | 3/5 [01:21<00:52, 26.39s/it]

## 6) Extract Column-Wise Data with LLM Structured Parsing

The previous cell already performs schema-driven JSON extraction with LLM (`llm_extract`) and deterministic fallback when LLM is unavailable.

## 7) Assemble Final DataFrame for All Columns

Conflict resolution rule used:
- If extraction confidence >= `min_confidence_to_update`, use extracted value.
- Otherwise retain original value.

Extra metadata columns are appended for transparency.

## 8) Render the Data in a Table Cell Output

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)

display(enriched_df)
print("Columns in final table:")
print(list(enriched_df.columns))

## 9) Caching, Retries, and Rate-Limit Handling

Implemented in the run cell via:
- `tenacity` retries with exponential backoff
- On-disk JSON cache for search and LLM results
- Per-batch checkpoint CSV for recovery
- Small sleep between requests to reduce rate-limit risk

## 10) Save Enriched Results and Artifacts

In [ ]:
Path(CONFIG["output_csv"]).parent.mkdir(parents=True, exist_ok=True)

enriched_df.to_csv(CONFIG["output_csv"], index=False)
try:
    enriched_df.to_parquet(CONFIG["output_parquet"], index=False)
except Exception as exc:
    print(f"Parquet save skipped: {exc}")

with open(CONFIG["evidence_json"], "w", encoding="utf-8") as f:
    json.dump(
        {
            "run_started_at": run_started_at,
            "run_finished_at": datetime.now(timezone.utc).isoformat(),
            "rows": len(enriched_df),
            "config": CONFIG,
            "evidence": evidence_log,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame(unresolved_rows).to_csv(CONFIG["unresolved_csv"], index=False)

print("Saved:")
print("-", CONFIG["output_csv"])
print("-", CONFIG["output_parquet"])
print("-", CONFIG["evidence_json"])
print("-", CONFIG["unresolved_csv"])
print("-", CONFIG["checkpoint_csv"])

summary_df = pd.DataFrame([
    {"metric": "total_rows", "value": len(enriched_df)},
    {"metric": "unresolved_rows", "value": len(unresolved_rows)},
    {"metric": "min_confidence_to_update", "value": CONFIG["min_confidence_to_update"]},
    {"metric": "used_serpapi", "value": bool(API_KEYS["serpapi"])},
    {"metric": "used_tavily_fallback", "value": bool(API_KEYS["tavily"])},
    {"metric": "used_llm", "value": bool(API_KEYS["openrouter"] and OpenAI is not None)},
])

display(summary_df)